# Notebook 02c: Gold Set Creation

**Project:** Hotel Curator -- Multi-Attribute Hotel Quality Classifier

**Inputs (from NB02a):**
- `data/audit/gold_set_template.csv` -- 200 reviews stratified by LF
  density, with empty annotation columns

**Output:**
- `data/audit/gold_set_annotated.csv` -- the same 200 reviews with all
  16 label columns filled in by an LLM (gpt-4o-mini / Grok 4.1
  Fast / Gemini 2.5 Flash-Lite, your choice)
- `data/audit/gold_set_audit.json` -- per-label distribution, agreement
  vs. silver labels, sample examples

**Why this notebook exists.** A "gold set" by convention is human-
annotated. For a take-home with no second annotator we have two
defensible options:
1. **Self-annotate** the 200-row CSV in Excel (~1 hour of careful work)
2. **LLM-annotate** with a stricter prompt + lower temperature than
   NB02b's relabeling pass, then human-spot-check a small sample

This notebook implements option 2. Treat the output as
**silver-plus** quality -- not true gold, but more careful than the
NB02b relabeling and explicitly designed for evaluation, not training.
The provenance (model, prompt hash, run timestamp) is persisted so the
report can disclose how the labels were produced.

---

## Differences from NB02b

| Dimension | NB02b (relabeling) | NB02c (gold) |
|---|---|---|
| Purpose | Augment training labels | Reference for evaluation |
| Reviews | 1,976 abstention pool | 200 stratified template |
| Prompt | Concise relabeling instructions | Detailed with rules + edge cases |
| Temperature | 0.1 | **0.0** (fully deterministic) |
| Model preference | Cost-optimized | Quality-optimized (still uses same key) |
| Concurrency | 5 | 5 |
| Cost | ~$0.10 | ~$0.02 (only 200 reviews) |

## Pipeline

0. Setup -- imports, config, seed, load template + reviews + taxonomy
1. Provider configuration (same env-key scheme as NB02b)
2. Stricter system prompt + Pydantic schema (matches NB02b's shape)
3. Annotation pass (concurrent sync calls)
4. Validation: parse rate, schema rate, per-label distribution
5. Compare against silver labels (agreement diagnostic, not gold-truth)
6. Persist `gold_set_annotated.csv` + audit JSON


In [1]:
# --- 0.0 Install dependencies (run once per environment) ---
# Already done if you followed env setup. Uncomment if needed.

# %pip install openai pydantic python-dotenv tqdm pandas pyarrow


In [2]:
# --- 0.1 Imports ---

import hashlib
import json
import os
import random
import re
import time
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from pydantic import BaseModel, Field, ValidationError
from tqdm.auto import tqdm

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

warnings.filterwarnings("ignore", category=UserWarning)


In [3]:
# --- 0.2 Configuration ---

SEED = 42

# Paths
DATA_DIR = Path("data")
PROCESSED_DIR = DATA_DIR / "processed"
AUDIT_DIR = DATA_DIR / "audit"

REVIEWS_PATH = PROCESSED_DIR / "reviews_clean.parquet"
SILVER_PATH = PROCESSED_DIR / "silver_labels_augmented.parquet"
TAXONOMY_PATH = PROCESSED_DIR / "attribute_taxonomy.json"
GOLD_TEMPLATE_PATH = AUDIT_DIR / "gold_set_template.csv"
GOLD_OUTPUT_PATH = AUDIT_DIR / "gold_set_annotated.csv"
GOLD_AUDIT_PATH = AUDIT_DIR / "gold_set_audit.json"
GOLD_RAW_PATH = AUDIT_DIR / "gold_set_raw_outputs.jsonl"

# LLM call settings -- stricter than NB02b
MAX_CONCURRENT = 5
MAX_RETRIES = 3
INITIAL_BACKOFF_SEC = 2.0
LLM_TEMPERATURE = 0.0   # deterministic for reference-quality labels

# How many to annotate. -1 = full template (recommended). Lower for
# a dry run.
N_REVIEWS_TO_ANNOTATE = -1

DEMO_MODE = False  # auto-engages if no API key

print("Config loaded.")
print(f"  GOLD_TEMPLATE_PATH = {GOLD_TEMPLATE_PATH}")
print(f"  GOLD_OUTPUT_PATH   = {GOLD_OUTPUT_PATH}")
print(f"  TEMPERATURE        = {LLM_TEMPERATURE}")


Config loaded.
  GOLD_TEMPLATE_PATH = data\audit\gold_set_template.csv
  GOLD_OUTPUT_PATH   = data\audit\gold_set_annotated.csv
  TEMPERATURE        = 0.0


In [4]:
# --- 0.3 Reproducibility ---


def set_seed(seed: int) -> None:
    """Set random seeds across libraries."""
    random.seed(seed)
    np.random.seed(seed)


set_seed(SEED)
print(f"Seed set: {SEED}")


Seed set: 42


In [5]:
# --- 0.4 Load template + reviews + taxonomy + silver ---


def load_artifacts(
    template_path: Path,
    reviews_path: Path,
    silver_path: Path,
    taxonomy_path: Path,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, list[str]]:
    """Load gold template, full reviews, silver labels, taxonomy."""
    for p in (template_path, reviews_path, silver_path, taxonomy_path):
        if not p.exists():
            raise FileNotFoundError(
                f"{p} missing. Run NB02a/b first."
            )
    template = pd.read_csv(template_path)
    reviews = pd.read_parquet(reviews_path)
    silver = pd.read_parquet(silver_path)
    with taxonomy_path.open() as f:
        labels = list(json.load(f)["labels"])
    return template, reviews, silver, labels


df_template, df_reviews, df_silver, ALL_LABELS = load_artifacts(
    GOLD_TEMPLATE_PATH, REVIEWS_PATH, SILVER_PATH, TAXONOMY_PATH,
)

print(f"Template:    {len(df_template):,} reviews to annotate")
print(f"Reviews:     {len(df_reviews):,} (full corpus)")
print(f"Silver:      {df_silver.shape}")
print(f"Labels:      {len(ALL_LABELS)}")

# Sanity: template review_ids must exist in the corpus
template_ids = set(df_template["review_id"])
corpus_ids = set(df_reviews["review_id"])
missing = template_ids - corpus_ids
if missing:
    raise ValueError(
        f"{len(missing)} template review_ids not in corpus: "
        f"{sorted(missing)[:5]}..."
    )
print("Template <-> corpus join: OK")


Template:    200 reviews to annotate
Reviews:     20,475 (full corpus)
Silver:      (20475, 17)
Labels:      16
Template <-> corpus join: OK


---

## Section 1: Provider Configuration

Same env-based provider selection as NB02b. Priority:
`OPENAI_API_KEY > XAI_API_KEY > GOOGLE_API_KEY`. We use the same
provider you used for relabeling -- the key already works.


In [6]:
# --- 1.1 Provider configuration ---


@dataclass(frozen=True)
class ProviderConfig:
    name: str
    model: str
    api_key_env: str
    base_url: str
    notes: str = ""


PROVIDER_PRESETS: dict[str, ProviderConfig] = {
    "openai": ProviderConfig(
        name="openai",
        model="gpt-4o-mini",
        api_key_env="OPENAI_API_KEY",
        base_url="https://api.openai.com/v1",
    ),
    "xai": ProviderConfig(
        name="xai",
        model="grok-4-1-fast-non-reasoning",
        api_key_env="XAI_API_KEY",
        base_url="https://api.x.ai/v1",
    ),
    "gemini": ProviderConfig(
        name="gemini",
        model="gemini-2.5-flash-lite",
        api_key_env="GOOGLE_API_KEY",
        base_url=(
            "https://generativelanguage.googleapis.com/v1beta/openai"
        ),
    ),
}


def select_provider() -> ProviderConfig | None:
    for name in ("openai", "xai", "gemini"):
        cfg = PROVIDER_PRESETS[name]
        key = os.environ.get(cfg.api_key_env)
        if not key and name == "gemini":
            key = os.environ.get("GEMINI_API_KEY")
        if key:
            return cfg
    return None


provider = select_provider()
if provider is None and not DEMO_MODE:
    print(
        "WARNING: no API key. Auto-engaging DEMO_MODE."
    )
    DEMO_MODE = True

if DEMO_MODE:
    print("DEMO_MODE = True; LLM calls will be SKIPPED.")
else:
    print(f"Selected provider: {provider.name}")
    print(f"  Model:    {provider.model}")
    print(f"  Endpoint: {provider.base_url}")


Selected provider: xai
  Model:    grok-4-1-fast-non-reasoning
  Endpoint: https://api.x.ai/v1


---

## Section 2: Schema and Strict Annotation Prompt

The schema is identical to NB02b -- 16 boolean fields. The system
prompt is **deliberately stricter**:
- Explicit rules about explicit-mention requirement
- Explicit handling of mixed-polarity cases ("spacious but old")
- Explicit handling of negation cues
- Explicit handling of vague/general praise
- Two anchor examples to ground the boundary cases


In [7]:
# --- 2.1 Output schema (matches NB02b) ---


class GoldReviewLabels(BaseModel):
    """Multi-hot label vector for one review."""

    cleanliness_positive: bool = Field(default=False)
    cleanliness_negative: bool = Field(default=False)
    room_positive: bool = Field(default=False)
    room_negative: bool = Field(default=False)
    staff_positive: bool = Field(default=False)
    staff_negative: bool = Field(default=False)
    location_positive: bool = Field(default=False)
    location_negative: bool = Field(default=False)
    noise_positive: bool = Field(default=False)
    noise_negative: bool = Field(default=False)
    wifi_positive: bool = Field(default=False)
    wifi_negative: bool = Field(default=False)
    value_positive: bool = Field(default=False)
    value_negative: bool = Field(default=False)
    breakfast_positive: bool = Field(default=False)
    breakfast_negative: bool = Field(default=False)


schema_fields = set(GoldReviewLabels.model_fields.keys())
assert schema_fields == set(ALL_LABELS), "Schema mismatch"
print(f"Schema validates: {len(schema_fields)} fields")


Schema validates: 16 fields


In [8]:
# --- 2.2 Strict annotation prompt ---

SYSTEM_PROMPT = """You are an expert hotel review annotator \
creating GOLD-STANDARD labels for a multi-label classification \
benchmark. Your output is treated as ground truth, so be careful, \
literal, and consistent.

For each review, identify which of the 8 hotel attributes are \
EXPLICITLY discussed and at what sentiment polarity.

THE 8 ATTRIBUTES:
- cleanliness: hygiene, dust, smells, sanitary state, condition of \
  bathrooms or surfaces
- room: comfort, size, condition, decor, beds, pillows, bathroom fixtures
- staff: service quality, friendliness, helpfulness, reception, concierge
- location: accessibility, neighborhood, distance to landmarks/transit
- noise: quietness, sleep quality, sound insulation, traffic/street noise
- wifi: internet quality, speed, availability, cost
- value: price perception, deals, hidden fees, worth-the-money assessment
- breakfast: food/F&B quality, variety, freshness, included or extra

STRICT ANNOTATION RULES:
1. Mark TRUE only if the review EXPLICITLY mentions that attribute \
   with that sentiment. The attribute must be discussed; sentiment \
   must be derivable from the text.
2. NEVER infer attributes that aren't mentioned. If a review never \
   discusses wifi, BOTH `wifi_positive` and `wifi_negative` are FALSE.
3. Vague generic praise like "great hotel overall" with no specific \
   attribute mentioned -> mark NOTHING.
4. Mixed reviews: a review CAN have BOTH polarities of the SAME \
   attribute (e.g., "spacious but old room" -> room_positive=true \
   AND room_negative=true). Do NOT use this rule as license to fire \
   both polarities when only one is clearly present.
5. Negation matters: "not clean" -> cleanliness_NEGATIVE (not pos). \
   "no noise" -> noise_POSITIVE (it's quiet). Read carefully.
6. Attribute boundaries:
   - "comfortable bed" -> room_positive (room comfort), not staff
   - "great breakfast" -> breakfast_positive only
   - "great location near transit" -> location_positive
   - "expensive parking" -> value_negative (price perception)
7. When unsure, default to FALSE. Precision over recall.

ANTI-INFERENCE RULES (these are the most common over-labeling traps):
- CLEANLINESS: "renovated", "modern", "brand new", "looked new", \
  "newly built" do NOT imply cleanliness_positive. Cleanliness must \
  be stated directly via words like "clean", "spotless", "tidy", \
  "fresh" or via condition language ("smelled", "dirty", "stained").
- ROOM: Listing amenities ("king bed", "balcony", "minifridge", \
  "view") without sentiment does NOT imply room_positive. Sentiment \
  words like "comfortable", "spacious", "cozy", "lovely", "nice room" \
  are required.
- LOCATION: Naming a city or neighborhood does NOT imply \
  location_positive. The reviewer must describe the location as \
  central, convenient, walkable, near things, or otherwise praise it.
- VALUE: Stating a price ("paid $200") or that a review was a \
  reasonable rate does NOT imply value_positive. The reviewer must \
  explicitly frame it as a deal, good value, or worth-the-money.
- BREAKFAST: A neutral mention ("had breakfast", "breakfast \
  included") is NOT breakfast_positive. Sentiment must be expressed \
  ("delicious", "great", "wonderful") or condition described \
  positively.
- GENERIC PRAISE: "great hotel", "lovely place", "perfect stay", \
  "loved it", "would return", "highly recommend" by themselves are \
  generic. Do NOT pick attributes from these alone.

EXAMPLE 1 (multi-attribute review):
Review: "great hotel, walking distance to metro, friendly staff, \
but the rooms are tiny and the wifi is slow"
Labels:
- staff_positive: true ("friendly staff")
- location_positive: true ("walking distance to metro")
- room_negative: true ("rooms are tiny")
- wifi_negative: true ("wifi is slow")
- ALL OTHERS: false (cleanliness, noise, value, breakfast not mentioned)

EXAMPLE 2 (vague generic praise -> nothing):
Review: "loved it, will be back"
Labels: ALL FALSE (no specific attributes mentioned)

EXAMPLE 3 (anti-inference trap):
Review: "stayed at this newly renovated boutique hotel in budapest \
for 3 nights. great experience overall, would recommend. king bed \
suite was lovely, breakfast included."
Labels:
- room_positive: true ("suite was lovely" — sentiment word)
- ALL OTHERS: false. Specifically:
  - cleanliness_positive: FALSE -- "newly renovated" is not cleanliness
  - location_positive: FALSE -- "budapest" is geographic, not praise
  - breakfast_positive: FALSE -- "breakfast included" is neutral
  - value_positive / staff_positive / noise_positive: FALSE -- not \
    discussed
  - "great experience overall" / "would recommend" are generic praise

OUTPUT: Return ONLY this JSON object, no preamble, no markdown fences:
{
  "cleanliness_positive": false, "cleanliness_negative": false,
  "room_positive": false, "room_negative": false,
  "staff_positive": false, "staff_negative": false,
  "location_positive": false, "location_negative": false,
  "noise_positive": false, "noise_negative": false,
  "wifi_positive": false, "wifi_negative": false,
  "value_positive": false, "value_negative": false,
  "breakfast_positive": false, "breakfast_negative": false
}"""

# Hash the prompt for provenance / reproducibility
PROMPT_HASH = hashlib.sha256(
    SYSTEM_PROMPT.encode("utf-8"),
).hexdigest()[:16]
print(f"System prompt: {len(SYSTEM_PROMPT)} chars")
print(f"Prompt hash:   {PROMPT_HASH}")


System prompt: 5092 chars
Prompt hash:   9818cfbedc36ffbf


---

## Section 3: LLM Annotation Pass

Same call mechanics as NB02b: parse + repair, retry with exponential
backoff, ThreadPoolExecutor concurrency. Differences:
- `temperature=0.0` (fully deterministic)
- Stricter prompt (above)
- Smaller workload (200 reviews vs 1,976)

Wall time: ~1-2 min depending on provider.


In [9]:
# --- 3.1 Parse helpers (reused pattern from NB02b) ---


def _normalize_label_value(v: object) -> bool:
    if isinstance(v, bool):
        return v
    if isinstance(v, int):
        return bool(v)
    if isinstance(v, str):
        return v.strip().lower() in {"true", "1", "yes", "y"}
    return False


def parse_llm_response(content: str) -> dict[str, bool] | None:
    if not content:
        return None
    s = content.strip()
    s = re.sub(r"^```(?:json)?\s*", "", s)
    s = re.sub(r"\s*```\s*$", "", s)
    match = re.search(r"\{.*\}", s, re.DOTALL)
    if not match:
        return None
    try:
        data = json.loads(match.group())
    except json.JSONDecodeError:
        return None
    if not isinstance(data, dict):
        return None
    return {
        label: _normalize_label_value(data.get(label, False))
        for label in ALL_LABELS
    }


def call_llm_label_review(
    client,
    model: str,
    text: str,
) -> tuple[dict[str, bool] | None, str]:
    """Call LLM, parse, validate. Returns (labels, raw_content)."""
    last_content = ""
    for attempt in range(MAX_RETRIES):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": text},
                ],
                response_format={"type": "json_object"},
                temperature=LLM_TEMPERATURE,
            )
            last_content = response.choices[0].message.content or ""
            parsed = parse_llm_response(last_content)
            if parsed is None:
                raise ValueError("parse failed")
            GoldReviewLabels.model_validate(parsed)
            return parsed, last_content
        except (ValueError, ValidationError, Exception) as e:
            last_content = f"{type(e).__name__}: {e}"
            if attempt == MAX_RETRIES - 1:
                return None, last_content
            time.sleep(INITIAL_BACKOFF_SEC * (2 ** attempt))
    return None, last_content


In [10]:
# --- 3.2 Self-test on one template review ---

if not DEMO_MODE:
    from openai import OpenAI

    client = OpenAI(
        api_key=os.environ.get(provider.api_key_env)
        or os.environ.get("GEMINI_API_KEY"),
        base_url=provider.base_url,
    )

    test_text = df_template.iloc[0]["text"]
    test_rid = int(df_template.iloc[0]["review_id"])
    print(f"Test review (rid={test_rid}):")
    print(f"  {test_text[:200]}...")
    print()

    labels, raw = call_llm_label_review(
        client, provider.model, test_text,
    )
    if labels is None:
        print("Self-test FAILED:")
        print(raw[:500])
    else:
        true_labels = [k for k, v in labels.items() if v]
        print(f"Self-test PASSED. {len(true_labels)} labels true:")
        for lbl in true_labels:
            print(f"  - {lbl}")
else:
    client = None
    print("DEMO_MODE: skipping self-test")


Test review (rid=1898):
  met expectation chose stay hotel excellent reviews read tripadvisor, not sure excellent reviews say inn average inn met expectation, staff excellent friendly staff* location average definitely not wal...

Self-test PASSED. 4 labels true:
  - cleanliness_positive
  - staff_positive
  - location_negative
  - value_positive


In [11]:
# --- 3.3 Concurrent annotation pass ---


def annotate_batch(
    client,
    model: str,
    texts: list[str],
    *,
    max_concurrent: int = MAX_CONCURRENT,
) -> tuple[list[dict[str, bool] | None], list[str]]:
    """Annotate a batch of texts concurrently."""
    results: list[dict[str, bool] | None] = [None] * len(texts)
    raws: list[str] = [""] * len(texts)
    with ThreadPoolExecutor(max_workers=max_concurrent) as ex:
        futures = {
            ex.submit(call_llm_label_review, client, model, t): i
            for i, t in enumerate(texts)
        }
        for fut in tqdm(
            as_completed(futures),
            total=len(futures),
            desc="Annotating",
        ):
            i = futures[fut]
            labels, raw = fut.result()
            results[i] = labels
            raws[i] = raw
    return results, raws


if N_REVIEWS_TO_ANNOTATE == -1:
    df_to_annotate = df_template.copy()
else:
    df_to_annotate = df_template.head(
        N_REVIEWS_TO_ANNOTATE
    ).reset_index(drop=True)
print(f"Reviews to annotate: {len(df_to_annotate):,}")

if DEMO_MODE:
    print("DEMO_MODE: producing all-None results.")
    annotations = [None] * len(df_to_annotate)
    raw_outputs = ["DEMO_MODE_SKIP"] * len(df_to_annotate)
else:
    t0 = time.time()
    annotations, raw_outputs = annotate_batch(
        client, provider.model, df_to_annotate["text"].tolist(),
    )
    elapsed = time.time() - t0
    print(f"Annotation: {len(df_to_annotate)} reviews in "
          f"{elapsed:.1f}s")


Reviews to annotate: 200


Annotating:   0%|          | 0/200 [00:00<?, ?it/s]

Annotation: 200 reviews in 71.9s


---

## Section 4: Validation, Audit, and Diagnostic Comparison

Three checks:
1. **Success rate** -- how many reviews produced parseable JSON
2. **Per-label distribution** -- what fraction of gold reviews carry
   each label; should look distributionally similar to the silver
   labels modulo the gold's stratified sampling
3. **Agreement vs silver labels** -- per-label agreement rate. This
   is NOT a model-quality metric (silver != truth), it's a diagnostic
   to spot prompt failures (e.g., if gold says wifi_positive=false on
   reviews where silver says true, the prompt may be too strict)


In [12]:
# --- 4.1 Validation summary ---

n_total = len(annotations)
n_success = sum(1 for x in annotations if x is not None)
n_fail = n_total - n_success
print("Annotation results:")
print(f"  total:    {n_total:,}")
print(f"  success:  {n_success:,} "
      f"({n_success / n_total * 100:.1f}%)")
print(f"  failed:   {n_fail:,} "
      f"({n_fail / n_total * 100:.1f}%)")

if n_success > 0:
    label_counts = {lbl: 0 for lbl in ALL_LABELS}
    for a in annotations:
        if a is not None:
            for lbl, v in a.items():
                if v:
                    label_counts[lbl] += 1
    print()
    print("Per-label gold positivity:")
    for lbl in sorted(
        label_counts, key=label_counts.get, reverse=True,
    ):
        c = label_counts[lbl]
        print(f"  {lbl:<26s} {c:>4,} "
              f"({c / n_success * 100:5.2f}%)")


Annotation results:
  total:    200
  success:  200 (100.0%)
  failed:   0 (0.0%)

Per-label gold positivity:
  staff_positive              138 (69.00%)
  location_positive           138 (69.00%)
  room_positive               131 (65.50%)
  cleanliness_positive        100 (50.00%)
  value_positive               61 (30.50%)
  room_negative                59 (29.50%)
  breakfast_positive           57 (28.50%)
  value_negative               44 (22.00%)
  noise_positive               41 (20.50%)
  staff_negative               34 (17.00%)
  cleanliness_negative         27 (13.50%)
  noise_negative               22 (11.00%)
  breakfast_negative           22 (11.00%)
  location_negative            14 ( 7.00%)
  wifi_positive                12 ( 6.00%)
  wifi_negative                 6 ( 3.00%)


In [13]:
# --- 4.2 Diagnostic agreement vs silver labels ---


def per_label_agreement(
    gold_records: list[dict[str, bool] | None],
    review_ids: list[int],
    silver: pd.DataFrame,
    label_names: list[str],
) -> dict[str, dict[str, float]]:
    """Per-label agreement rate (success cases only)."""
    silver_idx = silver.set_index("review_id")
    out: dict[str, dict[str, float]] = {}
    valid = [
        (rid, lab) for rid, lab in zip(review_ids, gold_records, strict=True)
        if lab is not None and rid in silver_idx.index
    ]
    n = len(valid)
    if n == 0:
        return out
    for lbl in label_names:
        agree = 0
        gold_pos = 0
        silver_pos = 0
        for rid, lab in valid:
            g = bool(lab[lbl])
            s = bool(silver_idx.at[rid, lbl])
            if g == s:
                agree += 1
            if g:
                gold_pos += 1
            if s:
                silver_pos += 1
        out[lbl] = {
            "agreement_rate": agree / n,
            "gold_positive_rate": gold_pos / n,
            "silver_positive_rate": silver_pos / n,
            "n": n,
        }
    return out


if n_success > 0:
    agreement = per_label_agreement(
        annotations,
        df_to_annotate["review_id"].tolist(),
        df_silver,
        ALL_LABELS,
    )
    print(f"  {'label':<26s} "
          f"{'agree':>7s} {'gold+':>7s} {'silver+':>8s}")
    print(f"  {'-' * 26} {'-' * 7} {'-' * 7} {'-' * 8}")
    for lbl in ALL_LABELS:
        m = agreement.get(lbl, {})
        if not m:
            continue
        print(
            f"  {lbl:<26s} "
            f"{m['agreement_rate'] * 100:>6.1f}% "
            f"{m['gold_positive_rate'] * 100:>6.1f}% "
            f"{m['silver_positive_rate'] * 100:>7.1f}%"
        )
else:
    agreement = {}
    print("No successful annotations; skipping agreement diagnostic.")


  label                        agree   gold+  silver+
  -------------------------- ------- ------- --------
  cleanliness_positive         91.5%   50.0%    49.5%
  cleanliness_negative         92.0%   13.5%     8.5%
  room_positive                71.0%   65.5%    42.5%
  room_negative                75.0%   29.5%     9.5%
  staff_positive               85.0%   69.0%    55.0%
  staff_negative               85.5%   17.0%     4.5%
  location_positive            66.0%   69.0%    39.0%
  location_negative            93.0%    7.0%     1.0%
  noise_positive               90.5%   20.5%    13.0%
  noise_negative               94.0%   11.0%     9.0%
  wifi_positive                97.5%    6.0%     4.5%
  wifi_negative                97.0%    3.0%     0.0%
  value_positive               76.0%   30.5%    13.5%
  value_negative               80.5%   22.0%     4.5%
  breakfast_positive           73.5%   28.5%     5.0%
  breakfast_negative           88.0%   11.0%     1.0%


In [14]:
# --- 4.3 Sample annotations for spot-check ---
# Two views:
#   (a) 3 random successful annotations (representative)
#   (b) 3 highest-disagreement-vs-silver annotations (boundary cases
#       — most informative for verifying the prompt is working)

if n_success > 0:
    success_idx = [
        i for i, x in enumerate(annotations) if x is not None
    ]
    rng = np.random.default_rng(SEED)

    # (a) Random sample
    random_pick = rng.choice(
        success_idx, size=min(3, len(success_idx)), replace=False,
    )

    # (b) Highest-disagreement sample
    silver_idx_df = df_silver.set_index("review_id")
    disagree_count: list[tuple[int, int]] = []
    for i in success_idx:
        rid = int(df_to_annotate.iloc[i]["review_id"])
        if rid not in silver_idx_df.index:
            disagree_count.append((i, 0))
            continue
        gold = annotations[i]
        n_diff = sum(
            1 for lbl in ALL_LABELS
            if bool(gold[lbl]) != bool(silver_idx_df.at[rid, lbl])
        )
        disagree_count.append((i, n_diff))
    disagree_count.sort(key=lambda x: x[1], reverse=True)
    high_disagree_pick = [i for i, _ in disagree_count[:3]]

    def _show(idx: int, header: str) -> None:
        rid = int(df_to_annotate.iloc[idx]["review_id"])
        text = df_to_annotate.iloc[idx]["text"]
        gold = annotations[idx]
        true_labels = [k for k, v in gold.items() if v]
        snippet = text[:240] + ("..." if len(text) > 240 else "")
        gold_only: list[str] = []
        silver_only: list[str] = []
        if rid in silver_idx_df.index:
            silver_row = silver_idx_df.loc[rid]
            gold_only = [
                lbl for lbl in ALL_LABELS
                if gold[lbl] and not silver_row[lbl]
            ]
            silver_only = [
                lbl for lbl in ALL_LABELS
                if silver_row[lbl] and not gold[lbl]
            ]
        print(f"{header} rid={rid}")
        print(f"  text: {snippet}")
        print(f"  gold labels: "
              f"{true_labels if true_labels else '(none)'}")
        if gold_only:
            print(f"  gold-only (silver missed):    {gold_only}")
        if silver_only:
            print(f"  silver-only (gold dropped):   {silver_only}")
        print()

    print("Random gold annotations (3 samples):")
    print("-" * 72)
    for idx in random_pick:
        _show(idx, "(random)        ")

    print("Highest-disagreement-vs-silver (3 samples):")
    print("-" * 72)
    for idx in high_disagree_pick:
        _show(idx, "(high-disagree) ")


Random gold annotations (3 samples):
------------------------------------------------------------------------
(random)         rid=18387
  text: great location service great deal, spent nights suites 1 king west week august 2006. generally visit toronto try save money staying suburban mississauga, spend visits toronto downtown core surrounding neighborhoods usually forgo downtown lo...
  gold labels: ['cleanliness_positive', 'room_positive', 'staff_positive', 'location_positive', 'value_positive']

(random)         rid=10119
  text: quiet relaxing hotel stayed double room hotel floor, room kept clean tidy folding pyjamas, shower fab just wish home, double bed huge comfortable plenty pillows, wide screen tele fixed opposite bed course nearly spanish, husband happy loads...
  gold labels: ['cleanliness_positive', 'room_positive', 'room_negative', 'location_positive', 'noise_positive']
  gold-only (silver missed):    ['room_negative']

(random)         rid=3900
  text: fabulous time great

---

## Section 5: Persist Gold Set + Audit

Two outputs:
- `gold_set_annotated.csv` -- the same shape as the template, with
  the 16 label columns filled with `0` or `1`. Failed annotations
  are left blank for human follow-up if you want to fix them.
- `gold_set_audit.json` -- model, prompt hash, timestamp, success
  rate, per-label distribution, agreement diagnostic.


In [15]:
# --- 5.1 Build the annotated CSV in template's shape ---

# Start from the template (preserves review_id, hotel_id, rating, text)
# Drop any pre-existing label columns from the template -- we'll rebuild them
# fresh as int dtype so PyArrow string dtype doesn't reject our writes.
keep_cols = ["review_id", "hotel_id", "rating", "text"]
df_gold = df_to_annotate[keep_cols].copy()

# Build the 16 label columns as a numeric matrix, then attach.
# -1 means "annotation failed / not filled"; 0/1 are real labels.
n_rows = len(df_to_annotate)
n_labels = len(ALL_LABELS)
label_matrix = np.full((n_rows, n_labels), -1, dtype=np.int8)

n_filled = 0
for i, lab in enumerate(annotations):
    if lab is None:
        continue
    for j, lbl in enumerate(ALL_LABELS):
        label_matrix[i, j] = int(lab[lbl])
    n_filled += 1

label_df = pd.DataFrame(
    label_matrix, columns=ALL_LABELS, index=df_gold.index,
)
df_gold = pd.concat([df_gold, label_df], axis=1)

# Replace -1 sentinels with empty strings ONLY at write time, so the
# downstream CSV reader (NB04 cell 8.1) can coerce blanks to 0/NaN
# without breaking on the int rows.
df_gold_out = df_gold.copy()
for lbl in ALL_LABELS:
    df_gold_out[lbl] = df_gold_out[lbl].astype(object)
    df_gold_out.loc[df_gold_out[lbl] == -1, lbl] = ""

df_gold_out.to_csv(GOLD_OUTPUT_PATH, index=False)

# Persist raw outputs for reproducibility
with GOLD_RAW_PATH.open("w", encoding="utf-8") as f:
    for rec, raw in zip(
        df_to_annotate.to_dict("records"), raw_outputs, strict=True,
    ):
        f.write(json.dumps({
            "review_id": int(rec["review_id"]),
            "text": rec["text"],
            "llm_raw_output": raw,
        }) + "\n")

print(f"Saved: {GOLD_OUTPUT_PATH} "
      f"({df_gold_out.shape}, {n_filled:,} rows annotated)")
print(f"Saved: {GOLD_RAW_PATH}")

Saved: data\audit\gold_set_annotated.csv ((200, 20), 200 rows annotated)
Saved: data\audit\gold_set_raw_outputs.jsonl


In [16]:
# --- 5.2 Persist audit ---

audit_payload: dict[str, object] = {
    "seed": SEED,
    "demo_mode": DEMO_MODE,
    "provider": (
        None if DEMO_MODE
        else {"name": provider.name, "model": provider.model}
    ),
    "system_prompt_hash": PROMPT_HASH,
    "system_prompt_chars": len(SYSTEM_PROMPT),
    "temperature": LLM_TEMPERATURE,
    "annotated_at": datetime.now(timezone.utc).isoformat(),
    "n_template": int(len(df_template)),
    "n_annotated": int(n_total),
    "n_success": int(n_success),
    "n_fail": int(n_fail),
    "success_rate": (
        float(n_success / n_total) if n_total else 0.0
    ),
    "per_label_gold_distribution": {
        lbl: int(label_counts[lbl]) if n_success > 0 else 0
        for lbl in ALL_LABELS
    },
    "agreement_vs_silver": agreement,
}
with GOLD_AUDIT_PATH.open("w") as f:
    json.dump(audit_payload, f, indent=2)
print(f"Saved: {GOLD_AUDIT_PATH}")

# Final summary block
print()
print("=" * 72)
print("GOLD SET CREATION COMPLETE")
print("=" * 72)
print(f"Reviews annotated: {n_success:,} / {n_total:,}")
if n_success > 0:
    avg_labels = float(np.mean([
        sum(a.values()) for a in annotations if a is not None
    ]))
    print(f"Avg labels per review: {avg_labels:.2f}")
    mean_agree = float(np.mean([
        m["agreement_rate"] for m in agreement.values()
    ]))
    print(f"Mean agreement vs silver: {mean_agree * 100:.1f}%")
print(f"Output: {GOLD_OUTPUT_PATH}")



Saved: data\audit\gold_set_audit.json

GOLD SET CREATION COMPLETE
Reviews annotated: 200 / 200
Avg labels per review: 4.53
Mean agreement vs silver: 84.7%
Output: data\audit\gold_set_annotated.csv


---

## Summary

| Artifact | Path | Description |
|---|---|---|
| Annotated gold set | `data/audit/gold_set_annotated.csv` | 200 reviews x 16 labels (0/1), LLM-annotated |
| Raw outputs | `data/audit/gold_set_raw_outputs.jsonl` | Reproducibility trail |
| Audit | `data/audit/gold_set_audit.json` | Provider, prompt hash, success rate, agreement |

### How to disclose this in REPORT.md

> The 200-review gold set was LLM-annotated (`<provider>/<model>`,
> temperature 0.0, system prompt hash `<hash>`). It is NOT
> human-annotated; for a take-home with no second annotator we treat
> it as silver-plus-quality reference labels rather than ground
> truth. Cohen's kappa is therefore not reported; agreement against
> the training silver labels is reported as a diagnostic only.

### When to upgrade to human gold

If you have time post-submission, manually review the
`gold_set_annotated.csv` for the 30-50 lowest-agreement rows
(where LLM gold disagrees with silver labels). These are the
boundary cases most likely to be misclassified, and human
re-annotation there has the highest marginal value.
